# 2. Features and leakage tests

**Question this notebook answers**
Does the corrected feature set still permit any exact reconstruction of the target? Answer should
be no, and the notebook should fail loudly rather than quietly if that ever stops being true.

**What v1 did**
`utilities/functions.py:196` writes `df[f"{col}_rollmean{window}"] = df[col].rolling(window).mean()`,
a trailing window inclusive of row `t`, and `utilities/functions.py:215` multiplies the raw target
into `Credit_Spread_x_FEDFUNDS`.

**Why that was wrong**
With `Credit_Spread_lag1` and `Credit_Spread_lag2` also present, the identity
`x(t) = 3*rollmean3(t) - x(t-1) - x(t-2)` makes the target an exact linear function of three of its
own features. `00_audit_of_v1.ipynb` shows the reconstruction error at the scale of floating-point
noise and an OLS fit on those three columns alone returning a test R-squared of one.

**What this notebook does instead**
Builds features only through `src/features.py`, where rolling windows are shifted and the shift is
part of the column name, then runs the `tests/` assertions inline so the notebook itself fails if
leakage ever returns.

**Predicted effect on results — recorded before running**
`leakage_report` should return an empty frame on the v2 feature set and a non-empty one on a
v1-style frame. The reconstruction error should be of the order of the spread's own standard
deviation, roughly 2 to 3, rather than near zero. No model built on these features should reach a
test R-squared anywhere near the v1 headline reproduced in `00_audit_of_v1.ipynb` section 1 —
expect roughly 0.6, which is what persistence
alone provides.

**Inputs** → **Outputs**
`data/v2/panel_pit.parquet` → in-memory feature frame consumed by notebooks 03 to 05; no files
written.

---
## 2.1 Load the point-in-time panel

Read `data/v2/panel_pit.parquet` and print its shape and date range.

---
## 2.2 Build the v2 feature frame

Call `src.features.build_feature_frame`. Print the target name it returns and the feature list. Look for the shift being visible in every rolling column name.

---
## 2.3 The reconstruction identity must fail

Call `src.features.rolling_identity_error` against the shifted rolling column. Look for an error of the order of the target's own standard deviation, not machine epsilon.

---
## 2.4 Run the leakage screen

Call `src.features.leakage_report` on the feature frame plus target. Assert the result is empty.

---
## 2.5 Show the screen fires on a v1-style frame

Build the v1 columns deliberately and run the same screen. An empty result on v2 means nothing unless the detector demonstrably fires on the known-bad case.

---
## 2.6 Run the test suite inline

Invoke pytest on `tests/test_features.py` from within the notebook and show the summary, so this notebook fails if the library regresses.

---
## 2.7 Stationarity of the feature set

Run ADF and KPSS on each feature. Flag any level series that both tests call non-stationary, and note which models are sensitive to that.

---
## 2.8 Feature summary for the record

Print the final feature list with, for each column, the timestamp of the newest observation it depends on. This table is the audit trail for every downstream model.